#인스톨

In [ ]:
!pip -q install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.8 MB/s eta 0:00:00


# 마운트

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 데이터 로드 및 파일 위치 저장

In [ ]:
import os
import yaml

## 본 코드는 절대경로로 되어 있습니다.
#### 실행시 파일 주소 설정 맞춰주세요
BASE_DIR = '/content/drive/MyDrive/AIKU_project_dent/AlphaDent'

# 데이터 루트 입니다. "위 베이스 경로에 맞춰서 들어가지므로 train val test 셋 위치를 파일에 맞춰주세요"
data_config = {
    'path': BASE_DIR,            'train': 'images/train',
    'val': 'images/valid',
    'test': 'images/test',

    # 클래스
    'names': {
        0: 'Abrasion',
        1: 'Filling',
        2: 'Crown',
        3: 'Caries Class 1',
        4: 'Caries Class 2',
        5: 'Caries Class 3',
        6: 'Caries Class 4',
        7: 'Caries Class 5',
        8: 'Caries Class 6'
    }
}


yaml_save_path = os.path.join(BASE_DIR, 'colab_config.yaml')

with open(yaml_save_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)
print('설정 완료')
print(f"저장 위치: {yaml_save_path}")

설정 완료
저장 위치: /content/drive/MyDrive/AIKU_project_dent/AlphaDent/colab_config.yaml


# 모델로드 (yolo v 11)

In [ ]:
from ultralytics import YOLO
import torch

# GPU 확인
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU세팅 : {gpu_name}")
else:
    print("런타임 CPU에 연결 -> GPU로 전환해주세요")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU세팅 : NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
from ultralytics import YOLO
import torch

# GPU 확인
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU세팅 : {gpu_name}")
else:
    print("런타임 CPU에 연결 -> GPU로 전환해주세요")

# 경로
data_yaml_path = '/content/drive/MyDrive/AIKU_project_dent/AlphaDent/colab_config.yaml'
project_path  = '/content/drive/MyDrive/AIKU_project_dent/AlphaDent/train_results'

# 모델 로드 (YOLOv11-seg 사용)
model = YOLO("yolo11m-seg.pt")


GPU세팅 : NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:

results = model.train(
    data=data_yaml_path,
    epochs=1200,
    patience=100,
    imgsz=1024,
    batch=16,
    device=0,
    workers=8,
    project=project_path,
    name='yolo11m_seg_1024_tuned_v2',
    exist_ok=True,

    optimizer="AdamW",
    lr0=1e-3,
    lrf=1e-2,
    weight_decay=0.01,
    warmup_epochs=5,
    cos_lr=True,

    augment=True,

    mosaic=0.5,
    close_mosaic=30,
    mixup=0.0,
    copy_paste=0.2,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.3,

    cache=True,
)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=30, cls=0.5, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/AIKU_project_dent/AlphaDent/colab_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m-seg.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolo11m_seg_1024_tuned_v2, nbs=64, nms

KeyboardInterrupt: 

In [ ]:
import os, glob, gc
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
from tqdm import tqdm
from torchvision.ops import nms as tv_nms

PROJECT_PATH    = "/content/drive/MyDrive/AIKU_project_dent/AlphaDent/train_results"
RUN_NAME        = "yolo11m_seg_1024_tuned_v2"
MODEL_PATH      = f"{PROJECT_PATH}/{RUN_NAME}/weights/best.pt"

TEST_IMAGES_DIR = "/content/drive/MyDrive/AIKU_project_dent/AlphaDent/images/test"
SUBMISSION_PATH = "/content/drive/MyDrive/AIKU_project_dent/AlphaDent/submission_stage1_ap50_tuned.csv"

assert os.path.exists(MODEL_PATH), f"model not found: {MODEL_PATH}"
assert os.path.isdir(TEST_IMAGES_DIR), f"test dir not found: {TEST_IMAGES_DIR}"

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

model = YOLO(MODEL_PATH)
print("model.names =", model.names)
name2id = {v: int(k) for k, v in model.names.items()}


#class th
CLASS_CONF_BY_NAME = {
    "Abrasion":       0.10,
    "Filling":        0.10,
    "Crown":          0.10,
    "Caries Class 1": 0.08,
    "Caries Class 2": 0.08,
    "Caries Class 3": 0.08,
    "Caries Class 4": 0.03,
    "Caries Class 5": 0.08,
    "Caries Class 6": 0.03,
}
CLASS_NMS_IOU_BY_NAME = {
    "Abrasion":       0.40,
    "Filling":        0.40,
    "Crown":          0.40,
    "Caries Class 1": 0.20,
    "Caries Class 2": 0.20,
    "Caries Class 3": 0.20,
    "Caries Class 4": 0.10,
    "Caries Class 5": 0.20,
    "Caries Class 6": 0.10,
}
DEFAULT_CONF = 0.10
DEFAULT_NMS_IOU = 0.45

CLASS_CONF = {name2id[n]: v for n, v in CLASS_CONF_BY_NAME.items() if n in name2id}
CLASS_NMS_IOU = {name2id[n]: v for n, v in CLASS_NMS_IOU_BY_NAME.items() if n in name2id}
print("CLASS_CONF =", CLASS_CONF)
print("CLASS_NMS_IOU =", CLASS_NMS_IOU)

#setting
PRED_IMGSZ     = 1024
RAW_CONF       = 0.02
RAW_IOU        = 0.65
RAW_MAX_DET    = 300
FINAL_MAX_DET  = 60

test_files = sorted(glob.glob(os.path.join(TEST_IMAGES_DIR, "*.jpg")))
print("num test images:", len(test_files))

#inference code
submission_data = []
kept_counter = defaultdict(int)

# 디버그 카운터
no_pred = 0   # 모델 출력이 0
no_keep = 0   # 후처리 후 0
no_poly = 0   # poly invalid로 버림

def log_gpu(tag=""):
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        used = total - free
        print(f"[GPU]{tag} used={used/1024**3:.2f}GB free={free/1024**3:.2f}GB total={total/1024**3:.2f}GB")

log_gpu(" start")

for k, file_path in enumerate(tqdm(test_files, total=len(test_files))):
    patient_id = os.path.splitext(os.path.basename(file_path))[0]

    results = model.predict(
        source=file_path,
        imgsz=PRED_IMGSZ,
        conf=RAW_CONF,
        iou=RAW_IOU,
        device=0,
        half=True,
        batch=1,
        retina_masks=True,
        max_det=RAW_MAX_DET,
        verbose=False,
    )

    r = results[0]
    if r.boxes is None or len(r.boxes) == 0 or r.masks is None:
        no_pred += 1
        if (k + 1) % 20 == 0:
            del results, r
            gc.collect()
            torch.cuda.empty_cache()
            log_gpu(f" after {k+1}")
        continue

    boxes_xyxy = r.boxes.xyxy.detach().cpu()
    boxes_conf = r.boxes.conf.detach().cpu()
    boxes_cls  = r.boxes.cls.detach().cpu().to(torch.int64)
    polys_xyn  = r.masks.xyn

    keep_indices = []
    for class_id in boxes_cls.unique().tolist():
        class_id = int(class_id)
        idx = torch.where(boxes_cls == class_id)[0]
        if len(idx) == 0:
            continue

        conf_th = CLASS_CONF.get(class_id, DEFAULT_CONF)
        idx = idx[boxes_conf[idx] >= conf_th]
        if len(idx) == 0:
            continue

        nms_iou = CLASS_NMS_IOU.get(class_id, DEFAULT_NMS_IOU)
        keep_local = tv_nms(boxes_xyxy[idx], boxes_conf[idx], nms_iou)
        idx = idx[keep_local]
        keep_indices.extend(idx.tolist())

    if not keep_indices:
        no_keep += 1
        if (k + 1) % 20 == 0:
            del results, r
            gc.collect()
            torch.cuda.empty_cache()
            log_gpu(f" after {k+1}")
        continue

    keep_indices = sorted(keep_indices, key=lambda i: float(boxes_conf[i]), reverse=True)[:FINAL_MAX_DET]

    for i in keep_indices:
        poly = polys_xyn[i]
        if poly is None or len(poly) < 3:
            no_poly += 1
            continue
        poly = np.clip(np.asarray(poly, dtype=np.float32), 0.0, 1.0)
        class_id = int(boxes_cls[i].item())
        confidence = float(boxes_conf[i].item())
        poly_str = " ".join(f"{v:.6f}" for v in poly.reshape(-1))
        submission_data.append([patient_id, class_id, confidence, poly_str])
        kept_counter[class_id] += 1

    if (k + 1) % 20 == 0:
        del results, r
        gc.collect()
        torch.cuda.empty_cache()
        log_gpu(f" after {k+1}")

df = pd.DataFrame(submission_data, columns=["patient_id", "class_id", "confidence", "poly"])
df = df.sort_values(["patient_id","class_id","confidence"], ascending=[True,True,False]).reset_index(drop=True)
df.insert(0, "id", range(len(df)))
df.to_csv(SUBMISSION_PATH, index=False)

print("saved:", SUBMISSION_PATH, "rows:", len(df))
print("kept per class:", dict(kept_counter))
print("no_pred (model outputs 0):", no_pred)
print("no_keep (postprocess -> 0):", no_keep)
print("no_poly (invalid poly skipped):", no_poly)

log_gpu(" end")

model.names = {0: 'Abrasion', 1: 'Filling', 2: 'Crown', 3: 'Caries Class 1', 4: 'Caries Class 2', 5: 'Caries Class 3', 6: 'Caries Class 4', 7: 'Caries Class 5', 8: 'Caries Class 6'}
CLASS_CONF = {0: 0.1, 1: 0.1, 2: 0.1, 3: 0.08, 4: 0.08, 5: 0.08, 6: 0.03, 7: 0.08, 8: 0.03}
CLASS_NMS_IOU = {0: 0.4, 1: 0.4, 2: 0.4, 3: 0.2, 4: 0.2, 5: 0.2, 6: 0.1, 7: 0.2, 8: 0.1}
num test images: 135
[GPU] start used=1.69GB free=93.29GB total=94.97GB


 16%|█▌        | 21/135 [00:08<00:42,  2.68it/s]

[GPU] after 20 used=2.29GB free=92.68GB total=94.97GB


 30%|███       | 41/135 [00:13<00:19,  4.88it/s]

[GPU] after 40 used=1.24GB free=93.73GB total=94.97GB


...

[GPU] after 100 used=4.10GB free=90.87GB total=94.97GB


 89%|████████▉ | 120/135 [00:41<00:05,  2.63it/s]

[GPU] after 120 used=4.10GB free=90.87GB total=94.97GB


100%|██████████| 135/135 [00:47<00:00,  2.87it/s]

saved: /content/drive/MyDrive/AIKU_project_dent/AlphaDent/submission_stage1_ap50_tuned.csv rows: 2503
kept per class: {1: 494, 7: 309, 0: 930, 5: 133, 2: 146, 4: 313, 3: 150, 8: 8, 6: 20}
no_pred (model outputs 0): 0
no_keep (postprocess -> 0): 0
no_poly (invalid poly skipped): 0
[GPU] end used=30.21GB free=64.76GB total=94.97GB


In [ ]:
from google.colab import files
files.download(SUBMISSION_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>